# 01 · Configuração do catálogo

Cria no Unity Catalog a estrutura do MVP: o catálogo `mvp_reclamacoes`, um schema por camada da arquitetura medalhão (`bronze`, `silver`, `gold`) e o volume `mvp_reclamacoes.bronze.arquivos`, que recebe os arquivos gerados por `scripts/baixar_reclamacoes.py`.

Todos os comandos usam `IF NOT EXISTS`, então rodar de novo não recria nem apaga nada.

**Ordem de execução**
1. Seções 1 a 3: criar e conferir a estrutura.
2. Upload manual pelo Catalog Explorer: os 68 `.csv.gz` de `dados/preparados/` vão para a pasta `reclamacoes/` do volume, e o `manifesto.csv` vai para a raiz do volume.
3. Seção 4: conferir o upload.

## 1. Catálogo e schemas

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS mvp_reclamacoes
COMMENT 'MVP de Engenharia de Dados (PUC): reclamações contra bancos e instituições de pagamento registradas no consumidor.gov.br, de jan/2021 a ago/2026.'

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS mvp_reclamacoes.bronze
COMMENT 'Camada bronze: dados como vieram da fonte, sem transformação, com metadados de ingestão. Guarda também o volume com os arquivos brutos.'

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS mvp_reclamacoes.silver
COMMENT 'Camada silver: dados limpos, tipados e padronizados.'

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS mvp_reclamacoes.gold
COMMENT 'Camada gold: modelo dimensional e agregações que respondem às perguntas de negócio.'

## 2. Volume dos arquivos brutos

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS mvp_reclamacoes.bronze.arquivos
COMMENT 'Arquivos mensais de reclamações finalizadas do consumidor.gov.br (Senacon/MJ, licença CC BY), em .csv.gz na pasta reclamacoes/, e manifesto.csv com a origem de cada arquivo. Gerados por scripts/baixar_reclamacoes.py.'

## 3. Conferência da estrutura

Esperado: os schemas `bronze`, `silver` e `gold` (além dos schemas de sistema) e o volume `arquivos` em `bronze`.

In [ ]:
%sql
SHOW SCHEMAS IN mvp_reclamacoes

In [ ]:
%sql
SHOW VOLUMES IN mvp_reclamacoes.bronze

## 4. Conferência do upload (rodar depois do upload)

Esperado:
- na pasta `reclamacoes/`, 68 linhas, de `finalizadas_2021-01.csv.gz` a `finalizadas_2026-08.csv.gz`;
- na raiz do volume, `manifesto.csv` e a pasta `reclamacoes/`.

In [ ]:
%sql
LIST '/Volumes/mvp_reclamacoes/bronze/arquivos/reclamacoes'

In [ ]:
%sql
LIST '/Volumes/mvp_reclamacoes/bronze/arquivos'

In [ ]:
# Conta os arquivos da pasta reclamacoes/ (o LIST acima pode exibir só as primeiras linhas).
# Esperado: 68 arquivos, 321.083.956 bytes (total de dados/preparados/*.csv.gz), de 2021-01 a 2026-08.
arquivos = dbutils.fs.ls("/Volumes/mvp_reclamacoes/bronze/arquivos/reclamacoes")
nomes = sorted(a.name for a in arquivos)
print(f"{len(arquivos)} arquivos | {sum(a.size for a in arquivos):,} bytes | {nomes[0]} ... {nomes[-1]}")